# **Optimal number of cells and virions to use**

Run the cell below to setup the folder with an example txt file with the abundances. Once the folder is setup, download the txt file and input the abundances following the same format and reupload while deleting the original txt file.

In [ ]:
!git clone https://github.com/jasoncngo/CRISPR-Screen-Equations.git
%cd CRISPR-Screen-Equations

Cloning into 'CRISPR-Screen-Equations'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 29 (delta 13), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 11.73 KiB | 480.00 KiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/CRISPR-Screen-Equations


The cell below calculated the amount of cells you need to use for the screen based on the abundance txt file and generates a lorenz plot of the guide abundance distribution.

In [ ]:
!python cellcount.py

The cell below calculate the amount of viral titer to add.

In [ ]:
!python viruscount.py

# **FACS analysis**


The cell below makes the files to input your data

In [ ]:
!mkdir -p XGBoost/train/R{1..6}/anchor
!mkdir -p XGBoost/predict/R{1..6}

Drag your control fcs files into train/R1 and baseline control into train/R1/anchor. Then input the markers you would like to seperate your data on.

In [ ]:
markers = ['FITC-A','FITC-W','FITC-H']

The cell below does the analysis

In [ ]:
!find . -maxdepth 4 -depth -type d -empty -delete
!pip install fcsparser --no-deps
!pip install numpy==1.26.4
import fcsparser
import pandas as pd
import numpy as np
import plotly.express as px
import os
from scipy.stats import gaussian_kde
import xgboost as xgb
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import RobustScaler, LabelEncoder
def calculate_fdr_threshold(y_true_encoded, prob_matrix, target_fdr=0.05):
    target_precision = 1 - target_fdr
    max_train_probs = prob_matrix.max(axis=1)
    train_predictions = prob_matrix.argmax(axis=1)
    is_correct = (train_predictions == y_true_encoded).astype(int)
    precisions, recalls, thresholds = precision_recall_curve(is_correct, max_train_probs)
    idx = np.where(precisions >= target_precision)[0]
    if len(thresholds)==idx[0]:
        return thresholds[idx[0]-1]
    if len(idx) > 0:
        return thresholds[idx[0]]
    else:
        return thresholds[-1]
def apply_fixed_shift(target_df, source_df):
    res = target_df.copy()
    for col in target_df.columns:
        source_data = source_df[col].values.flatten()
        jitter = np.random.normal(0, 1e-10, len(source_data))
        kde = gaussian_kde(source_data + jitter)
        x_range = np.linspace(source_data.min(), source_data.max(), 1000)
        peak = x_range[np.argmax(kde(x_range))]
        res[col] = target_df[col] - peak
    return res
def clean_batch_names(names):
    names2=[f for f in names if not f.startswith('.')]
    if len(names2)==1:
        return names2
    if not names2: return []
    prefix_len = 0
    for chars in zip(*names2):
        if len(set(chars)) == 1:
            prefix_len += 1
        else:
            break
    suffix_len = 0
    for chars in zip(*[n[::-1] for n in names2]):
        if len(set(chars)) == 1:
            suffix_len += 1
        else:
            break
    return [n[prefix_len : len(n)-suffix_len] for n in names]
def analyze(path,predict):
    base_path, scalelist='/content/XGBoost/'+path, {}
    dirlist=[f for f in os.listdir(base_path) if not f.startswith('.') and os.path.isdir(os.path.join(base_path, f)) and any(not item.startswith('.') for item in os.listdir(os.path.join(base_path, f)))]
    for m in range(len(dirlist)):
        dirloc=(base_path+'/'+dirlist[m])
        filenames=[f for f in os.listdir(dirloc) if not f.startswith('.')]
        if 'anchor' in filenames:
            filenames.remove("anchor")
        anchorname=[f for f in os.listdir('/content/XGBoost/train/'+dirlist[m]+'/anchor') if not f.startswith('.')][0]
        for k in range(len(filenames)):
            file_path = os.path.join(dirloc, filenames[k])
            if os.path.isfile(file_path):
                if '.fcs' in file_path:
                    data=fcsparser.parse(file_path, channel_naming="$PnN")[1].select_dtypes(['number'])[markers]
                if '.csv' in file_path:
                    data=pd.read_csv(file_path).select_dtypes(['number']).dropna(axis=0)[markers]
                if predict=='prediction':
                    data=pd.DataFrame(model.predict_proba(scaler.transform(apply_fixed_shift(data,scalelistt[dirlist[m]]))), columns=le.classes_)
                    labels = np.where(data.max(axis=1) >= threshold, data.idxmax(axis=1), 'Ungated')
                    data=pd.DataFrame(pd.Series(labels).value_counts(normalize=True))
                if path=='train':
                    cleannames=clean_batch_names(filenames+[anchorname])
                    cleananchorname=cleannames[len(filenames)]
                    data['Sample']=cleannames[k]
                if path=='predict':
                    data['Sample']=clean_batch_names(filenames)[k]
                if k>0:
                    fulldata=pd.concat([fulldata, data], axis=0)
                if k==0 and predict!='prediction':
                    fulldata=fcsparser.parse(dirloc+'/anchor/'+anchorname, channel_naming="$PnN")[1].select_dtypes(['number'])[markers]
                    fulldata['Sample']=clean_batch_names([anchorname]+filenames)[0]
                    anchordata=fulldata
                    fulldata=pd.concat([fulldata, data], axis=0)
                if k==0 and predict=='prediction':
                    if path=='train':
                        padata=fcsparser.parse(dirloc+'/anchor/'+anchorname, channel_naming="$PnN")[1].select_dtypes(['number'])[markers]
                        adata=pd.DataFrame(model.predict_proba(scaler.transform(apply_fixed_shift(padata,scalelistt[dirlist[m]]))), columns=le.classes_)
                        alabels = np.where(adata.max(axis=1) >= threshold, adata.idxmax(axis=1), 'Ungated')
                        adata=pd.DataFrame(pd.Series(alabels).value_counts(normalize=True))
                        adata['Sample']=clean_batch_names([anchorname]+filenames)[0]
                        fulldata=pd.concat([adata, data], axis=0)
                    if path=='predict':
                        fulldata=data
        if path=='predict' or predict=='prediction':
            if m>0:
                fulldatall=pd.concat([fulldatall, fulldata], axis=0)
            if m==0:
                fulldatall=fulldata
        if predict!='prediction':
            scalelist[dirlist[m]]=anchordata
            cordata=apply_fixed_shift(fulldata[markers],anchordata)
            cordata['Sample']=list(fulldata['Sample'])
            if m>0:
                fulldatall=pd.concat([fulldatall, cordata], axis=0)
            if m==0:
                fulldatall=cordata
    if path=='predict' or predict=='prediction':
        return (fulldatall)
    if path=='train' and predict!='prediction':
        return ([fulldatall,scalelist,cleananchorname])
scaler, le = RobustScaler(), LabelEncoder()
traina=analyze('train','noprediction')
fulldata, scalelistt=traina[0], traina[1]
X_train_raw, y_train_encoded=fulldata.drop('Sample', axis=1), le.fit_transform(fulldata['Sample'])
X_train_scaled, model=scaler.fit_transform(X_train_raw), xgb.XGBClassifier(random_state=42)
model.fit(X_train_scaled, y_train_encoded)
train_probs = model.predict_proba(X_train_scaled)
threshold = calculate_fdr_threshold(y_train_encoded, train_probs, target_fdr=0.05)
foldercount=len([f for f in os.listdir('/content/XGBoost/predict/R1') if not f.startswith('.')])
if foldercount>0:
    final=pd.concat([analyze('predict', 'prediction'), analyze('train', 'prediction')], axis=0)
if foldercount==0:
    final=analyze('train', 'prediction')
!mkdir XGBoost/output
final.to_csv('/content/XGBoost/output/ratio.csv')

# **CRISPRtile prediction**

1. Make sure you have Colab Pro+ so the notebook doesn't disconnect in the middle of the analysis.

2. Run the cell below which prepares the folder for the analysis. It connects to google drive and creates a folder called 'CRISPRtile' to save the results before it disconnects and deletes the data.

3. Once the CRISPRtile folder is generated in your google drive, drag all your .csv and .pdb files into the 'CRISPRtile/input' folder located in your google drive. pdb files of the proteins you are targeting can be found here https://alphafold.ebi.ac.uk/. A runnable folder setup can be found in the CRISPRtile folder in this colab after running the below cell where you can replicate the analysis from the paper by dragging all the files in the 'CRISPRtile/input' folder in this colab to the one in your google drive.

In [ ]:
!git clone https://github.com/jasoncngo/CRISPRtile.git
from google.colab import drive
import os
drive.mount('/content/gdrive', force_remount=True)
if os.path.exists("gdrive/MyDrive/CRISPRtile") == False:
    os.mkdir("gdrive/MyDrive/CRISPRtile")
    os.mkdir("gdrive/MyDrive/CRISPRtile/input")
    os.mkdir("gdrive/MyDrive/CRISPRtile/output")

Cloning into 'CRISPRtile'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 43 (delta 7), reused 24 (delta 4), pack-reused 16 (from 1)
Receiving objects: 100% (43/43), 1.17 GiB | 38.12 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (21/21), done.
Mounted at /content/gdrive


The cell below analyzes your data. The results will be saved in your google drive 'CRISPRtile/outputs' folder. It will take about a day to complete running.

In [ ]:
import glob
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib
def findsgrnacolumn(dataframe):
    findsgrnacolumn=list(dataframe.T[0])
    for k in range(len(findsgrnacolumn)):
        if type(findsgrnacolumn[k])==str and len(findsgrnacolumn[k])==20:
            counter=0
            for m in range(len(findsgrnacolumn[k])):
                if findsgrnacolumn[k][m] not in ['A','T','C','G']:
                    counter=1
                if m==19 and counter==0:
                    return(k)
inputdf=pd.read_csv('gdrive/MyDrive/CRISPRtile/input/input.csv')
sgrnacolumn=findsgrnacolumn(inputdf)
scoreheader, guideheader=list(inputdf.columns)[sgrnacolumn+1:], list(inputdf.columns)[:sgrnacolumn]
if glob.glob('gdrive/MyDrive/CRISPRtile/output/annotation.csv')!=['gdrive/MyDrive/CRISPRtile/output/annotation.csv']:
    print('Annotation file has not been generated yet in output folder. Generating annotation file. ')
    !pip install mdtraj==1.11.0
    import mdtraj as md
    import gzip
    import csv
    def analyzepdb(file):
        pdb = md.load(pdbfiles[file])
        dssps, dssp = list(md.compute_dssp(pdb, simplified=False)[0]), list(md.compute_dssp(pdb)[0])
        ssd=pd.DataFrame(zip(dssp,dssps),index=[(k+1) for k in range(len(dssp))], columns=['Secondary Structure','Secondary Structure Detailed'])
        sasa = md.shrake_rupley(pdb, mode = 'residue')[0]
        sasadf=pd.DataFrame(sasa,index=[(k+1) for k in range(len(sasa))],columns=['Solvent-Accessible Surface Area'])
        d = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL':'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}
        pdbdf=pdb.topology.to_dataframe()[0].replace({"resName": d}).join(pd.DataFrame(pdb.xyz[0], columns=['X_coord','Y_coord','Z_coord']))
        dfxyz=pdbdf[pdbdf['name'].isin(['N','CA','C','O'])]
        atoms=list(set(dfxyz['name']))
        header=['resSeq']+['resName']+[atoms[k]+'_x' for k in range(len(atoms))]+[atoms[k]+'_y' for k in range(len(atoms))]+[atoms[k]+'_z' for k in range(len(atoms))]
        dfxyzvalues, finalxyz, tempxyz=dfxyz.values.tolist(), [], [0 for k in range(len(header))]
        previd=dfxyzvalues[0][3]
        for k in range(len(dfxyzvalues)):
            if dfxyzvalues[k][3]!=previd:
                finalxyz.append(tempxyz)
                tempxyz=[0 for k in range(len(header))]
            previd, tempxyz[0], tempxyz[1]=dfxyzvalues[k][3], dfxyzvalues[k][3], dfxyzvalues[k][4]
            tempxyz[header.index(dfxyzvalues[k][1]+'_x')], tempxyz[header.index(dfxyzvalues[k][1]+'_y')], tempxyz[header.index(dfxyzvalues[k][1]+'_z')]=dfxyzvalues[k][8], dfxyzvalues[k][9], dfxyzvalues[k][10]
            if k==len(dfxyzvalues)-1:
                finalxyz.append(tempxyz)
        pdbanalyzed=pd.DataFrame(finalxyz, columns=header).sort_values(by=['resSeq']).set_index('resSeq').join(ssd).join(sasadf)
        return(pdbanalyzed)
    pdbfiles, aminolist, pdbdfmatrix=glob.glob('gdrive/MyDrive/CRISPRtile/input/*.pdb'), [], []
    for k in range(len(pdbfiles)):
        dfanalyzepdb=analyzepdb(k)
        pdbdfmatrix.append(dfanalyzepdb)
        aminolist.append(list(dfanalyzepdb['resName'])+['*'])
    geneid=[[] for k in range(len(aminolist))]
    def annotation():
        flist, prev, prevAA, aminodone=[], 0, 0, []
        for k in range(1,14):
            with gzip.open('CRISPRtile/annotation/annotation'+str(k)+'.csv.gz', mode='rt') as file:
                reader = csv.reader(file)
                if k==1:
                    for line in reader:
                        if 'ensembl_peptide_id' in line:
                            columnlabel=line
                            break
                for line in reader:
                    if line[2]!=prev:
                        templist, tempamino=[line], [line[7]]
                    if line[2]==prev:
                        templist.append(line)
                        if line[6]!=prevAA:
                            tempamino.append(line[7])
                    prev, prevAA=line[2], line[6]
                    if tempamino in aminolist:
                        geneid[aminolist.index(tempamino)].append(line[4])
                        flist+=templist
                        print(line[4]+' annotation is complete')
                        if tempamino not in aminodone:
                            aminodone.append(tempamino)
                            if len(aminodone)==len(aminolist):
                                return(pd.DataFrame(flist,columns=columnlabel))
                        templist, tempamino=[], []
            print(str(7.6*k)+'% complete')
    annotation1, masterlistmatrix = annotation(), []
    for k in range(len(geneid)):
        pdbannotation=annotation1[annotation1['transcript_name'].isin([geneid[k][0]])].copy().reset_index()
        pdbannotation['index']=pdbannotation['position'].astype(int)
        masterlistmatrix.append(pdbannotation.set_index('index').join(pdbdfmatrix[k]))
    premlist=pd.concat([masterlistmatrix[k] for k in range(len(masterlistmatrix))]).reset_index()
    premlist['index']=premlist['guide']
    mlist1=premlist.set_index('index')
    mlist2=mlist1.join(inputdf.set_index(list(inputdf.columns)[findsgrnacolumn(inputdf)]))
    listsecstrucdetailed, secstrucnew=list(mlist2['Secondary Structure Detailed']), []
    for k in range(len(listsecstrucdetailed)):
        if listsecstrucdetailed[k]==' ':
            secstrucnew.append('N')
        else:
            secstrucnew.append(listsecstrucdetailed[k])
    mlist2['Secondary Structure Detailed'], mlist2['SecStruct']=secstrucnew, mlist2['Secondary Structure'].replace(['E'], 'B')
    mlist2=mlist2.sort_values(by=['transcript_name','gene_fraction']).reset_index().drop(columns=['index','resName', 'Secondary Structure']).dropna(subset=['Solvent-Accessible Surface Area'])
    MLprep=mlist2.filter(items=['transcript_name','AA', 'position', 'offtarget_score', 'doench_score', 'oof_score', 'provean_score', 'disorder_score']
                     +((list(mlist2.columns)[list(mlist2.columns).index('control')+1:])))
    onehotindex, transcript, AA, secstruc=list(MLprep.index), MLprep['transcript_name'], MLprep['AA'], MLprep['Secondary Structure Detailed']
    transcriptset, AAset, secstrucset=list(set(transcript)), list(set(AA)), list(set(secstruc))
    onehotset=transcriptset+['AA-'+str(AAset[k]) for k in range(len(AAset))]+['SS-'+str(secstrucset[k]) for k in range(len(secstrucset))]
    zeromatrix=[0 for k in range(len(onehotset))]
    tempzero, onehotlist=zeromatrix.copy(), []
    for k in range(len(AA)):
        if list(transcript)[k] in onehotset:
            tempzero[onehotset.index(list(transcript)[k])]=1
        if 'AA-'+list(AA)[k] in onehotset:
            tempzero[onehotset.index('AA-'+list(AA)[k])]=1
        if 'SS-'+list(secstruc)[k] in onehotset:
            tempzero[onehotset.index('SS-'+list(secstruc)[k])]=1
        onehotlist.append([onehotindex[k]]+tempzero)
        tempzero=zeromatrix.copy()
    onehotdf=pd.DataFrame(onehotlist, columns=['numberindex']+onehotset).set_index('numberindex')
    MLfinal=(onehotdf.join(MLprep)).drop(columns=["transcript_name","AA","Secondary Structure Detailed"])
    MLfinal.to_csv('gdrive/MyDrive/CRISPRtile/output/train.csv')
    mlist2.to_csv('gdrive/MyDrive/CRISPRtile/output/annotation.csv')
    print('Annotation file has been saved. You can resume progress from this point by running the program again.')
print('Generated annotation file has been found in the output folder. Resuming progress from this point. If you would like to start over, delete all files in the output folder.')
!pip install autogluon.tabular[all]==1.3.1
!pip install dask[dataframe]
from autogluon.tabular import TabularDataset, TabularPredictor
if glob.glob('gdrive/MyDrive/CRISPRtile/output/predict.csv')!=['gdrive/MyDrive/CRISPRtile/output/predict.csv']:
    import torch
    MLfinal=pd.read_csv('gdrive/MyDrive/CRISPRtile/output/train.csv').drop(columns=['numberindex'])
    MLfinal=MLfinal.replace([np.inf, -np.inf, 'NA'], np.nan)
    for k in range(len(scoreheader)):
        if glob.glob('gdrive/MyDrive/CRISPRtile/output/Models/'+scoreheader[k])!=['gdrive/MyDrive/CRISPRtile/output/Models/'+scoreheader[k]]:
            MLtrain=MLfinal.dropna(subset=[scoreheader[k]]).drop(columns=(scoreheader[:k]+scoreheader[k+1:]))
            predictor=TabularPredictor(label=scoreheader[k], path="gdrive/MyDrive/CRISPRtile/output/Models/"+scoreheader[k]).fit(MLtrain, presets="best_quality", time_limit=40000, num_gpus=torch.cuda.device_count())
    correctlist=guideheader+['offtarget_score', 'doench_score', 'oof_score']
    #correctlist=guideheader
    for k in range(len(correctlist)):
        if type(MLfinal[correctlist[k]].mode()[0])==str:
            MLfinal[correctlist[k]]=[MLfinal[correctlist[k]].mode()[0] for m in range(len(MLfinal))]
        if type(MLfinal[correctlist[k]].mode()[0])!=str:
            MLfinal[correctlist[k]]=[MLfinal[correctlist[k]].median() for m in range(len(MLfinal))]
    mlist2=pd.read_csv('gdrive/MyDrive/CRISPRtile/output/annotation.csv').drop(columns=['Unnamed: 0'])
    for k in range(len(scoreheader)):
        predictor = TabularPredictor.load("gdrive/MyDrive/CRISPRtile/output/Models/"+scoreheader[k])
        predictdf=pd.DataFrame(predictor.predict(MLfinal))
        predictdf.rename(columns = {scoreheader[k]:scoreheader[k]+' predict'}, inplace = True)
        mlist2=mlist2.join(predictdf)
    mlist2.to_csv('gdrive/MyDrive/CRISPRtile/output/predict.csv')
df=pd.read_csv('gdrive/MyDrive/CRISPRtile/output/predict.csv')
setlist=list(set(df['gene_name']))
for j in range(len(setlist)):
    df=pd.read_csv('gdrive/MyDrive/CRISPRtile/output/predict.csv')
    df=df[df['gene_name']==setlist[j]].drop_duplicates(subset=['position'])
    pred, position, changepoints=list(df['Function predict']), list(df['position']), [1]
    for k in range(len(pred)-1):
        if (pred[k]<np.median(pred) and pred[k+1]>np.median(pred)):
            changepoints.append(position[k])
        if (pred[k]>np.median(pred) and pred[k+1]<np.median(pred)):
            changepoints.append(position[k])
    changepoints.append(position[len(pred)-1])
    max, right=position[pred.index(np.min(pred))], 0
    for k in range(len(changepoints)):
        if changepoints[k]<max:
            left=changepoints[k]
        if right==0:
            if changepoints[k]>max:
                right=changepoints[k]
    halfmax, halfleft, n=(np.median(pred)+np.min(pred))/2, 0, 0
    while halfleft==0:
        curval=pred[position.index(left)+n]
        if curval<halfmax:
            if position[pred.index(curval)]!=np.min(position):
                halfleft=position[pred.index(curval)-1]
            else:
                halfleft=np.min(position)
        n+=1
    halfright, n =0, 0
    while halfright==0:
        curval=pred[position.index(right)-n]
        if curval<halfmax:
            if position[pred.index(curval)]!=np.max(position):
                halfright=position[pred.index(curval)+1]
            else:
                halfright=np.max(position)
        n+=1
    df=df[df['position']<=halfright]
    df=df[df['position']>=halfleft]
    pd.DataFrame([[str(halfleft)+'-'+str(halfright),''.join(list(df['AA']))]], columns=['label','sequence']).to_csv('gdrive/MyDrive/CRISPRtile/output/'+setlist[j]+'sequence.csv')
import plotly.express as px
from plotly import subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
mlist2=pd.read_csv('gdrive/MyDrive/CRISPRtile/output/predict.csv').drop(columns=['Unnamed: 0'])
mlist2=mlist2.replace(['#NAME?'], np.nan)
mlist2['Interpro_Description']=mlist2['Interpro_Description'].replace([np.nan], 'NA')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('N', 'Loops and irregular elements')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('H', 'Alpha helix')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('B', 'Residue in isolated beta-bridge')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('E', 'Extended strand, participates in beta ladder')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('G', '3-helix (3/10 helix)')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('I', '5 helix (pi helix)')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('T', 'Hydrogen bonded turn')
mlist2['Secondary Structure Detailed'] = mlist2['Secondary Structure Detailed'].str.replace('S', 'Bend')
for k in range(len(scoreheader)):
    mlist2[scoreheader[k]]=mlist2[scoreheader[k]].astype(float)
transcriptset, predictcol=list(set(mlist2['transcript_name'])), [scoreheader[m]+' predict' for m in range(len(scoreheader))]
parallelcollist=['position','provean_score','disorder_score','Solvent-Accessible Surface Area']+predictcol
for k in range(len(transcriptset)):
    plotdf=mlist2[mlist2['transcript_name'].isin([transcriptset[k]])]
    fig = go.Figure(data=go.Parcoords(line = dict(color = plotdf['position'],colorscale = px.colors.diverging.Tealrose),
        dimensions = [dict(label = parallelcollist[k], values = plotdf[parallelcollist[k]]) for k in range(len(parallelcollist))]),
                   layout = go.Layout(autosize=False,width=800,height=500, title=transcriptset[k]))
    fig.write_html('gdrive/MyDrive/CRISPRtile/output/'+transcriptset[k]+' parallelplot interactive.html')
    def pca(columns, whichpca):
        X = plotdf[columns]
        pca = PCA(n_components=2)
        components = pca.fit_transform(X)
        loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
        fig = px.scatter(components, x=0, y=1, color=plotdf[predictcol[m]], size=plotdf['Solvent-Accessible Surface Area'])
        for i, feature in enumerate(columns):
            fig.add_annotation(ax=0, ay=0,axref="x", ayref="y",x=loadings[i, 0],y=loadings[i, 1],showarrow=True,arrowsize=2,arrowhead=2,xanchor="right",yanchor="top")
            fig.add_annotation(x=loadings[i, 0],y=loadings[i, 1],ax=0, ay=0,xanchor="center", yanchor="bottom",text=feature,yshift=5)
        fig.update_layout(title={'text': transcriptset[k],'y':0.95,'x':0.5,'xanchor': 'center','yanchor': 'top'},
                          xaxis= {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': whichpca+'PC1 ('+str(round(pca.explained_variance_ratio_[0], 3)*100)+'% Variance)'}},
                         yaxis = {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': whichpca+'PC2 ('+str(round(pca.explained_variance_ratio_[1], 3)*100)+'% Variance)'}},
                         coloraxis = {'colorbar': {'title': {'text': scoreheader[m]}}})
        fig.write_html('gdrive/MyDrive/CRISPRtile/output/'+transcriptset[k]+' '+scoreheader[m]+' '+whichpca+'PCA interactive.html')
    for m in range(len(predictcol)):
        pca(['CA_x','CA_y','CA_z'], '3D ')
        pca(['provean_score','disorder_score','Solvent-Accessible Surface Area'], 'PDS ')
        violingroup=['AA','Interpro_Description','Secondary Structure Detailed']
        for n in range(len(violingroup)):
            fig = go.Figure()
            groups = set(plotdf[violingroup[n]])
            if n==0:
                for group in groups:
                    fig.add_trace(go.Violin(x=plotdf[violingroup[n]][plotdf[violingroup[n]] == group],y=plotdf[predictcol[m]][plotdf[violingroup[n]] == group],
                                    name=group,box_visible=True,meanline_visible=True, line_color='black',showlegend=False))
            else:
                for group in groups:
                    fig.add_trace(go.Violin(x=plotdf[violingroup[n]][plotdf[violingroup[n]] == group],y=plotdf[predictcol[m]][plotdf[violingroup[n]] == group],
                                    name=group,box_visible=True,meanline_visible=True))
            fig.update_layout(title={'text': transcriptset[k],'y':0.95,'x':0.5,'xanchor': 'center','yanchor': 'top'},
                                  xaxis= {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': violingroup[n]}},
                                 yaxis = {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': scoreheader[m]}})
            fig.write_html('gdrive/MyDrive/CRISPRtile/output/'+transcriptset[k]+' '+scoreheader[m]+' '+violingroup[n]+' violin interactive.html')
        fig2 = px.scatter(plotdf, x="position", y=predictcol[m], color="Interpro_Description", marginal_y="box", size="Solvent-Accessible Surface Area")
        fig2.update_yaxes(title_text=scoreheader[m], col=1)
        fig2.update_xaxes(title_text='Position', col=1)
        fig2.update_layout(title_text=transcriptset[k])
        fig2.write_html('gdrive/MyDrive/CRISPRtile/output/'+transcriptset[k]+' '+scoreheader[m]+' SASAsize interactive.html')
    position, exon, lineargroups=list(plotdf['position']), list(plotdf['Exon']), scoreheader+['provean_score','disorder_score']
    fig=subplots.make_subplots(rows=(len(lineargroups)),cols=1, vertical_spacing = 0.01)
    for t in range(len(lineargroups)):
        if lineargroups[t] in scoreheader:
            fig.add_trace(go.Scatter(x=position, y=plotdf[lineargroups[t]], name = lineargroups[t]+' raw', mode='markers', legendgroup = str(t+1)), row=t+1, col=1)
            predict=list(plotdf[lineargroups[t]+' predict'])
            fig.add_trace(go.Scatter(x=position, y=predict, name = lineargroups[t]+' guide efficiency corrected', legendgroup = str(t+1)), row=t+1, col=1)
        if lineargroups[t] == 'provean_score':
            setdomain=list(set(plotdf["Interpro_Description"]))
            for m in range(len(setdomain)):
                domainsubset=plotdf[plotdf['Interpro_Description'].isin([setdomain[m]])]
                fig.add_trace(go.Scatter(x=domainsubset["position"], y=domainsubset[lineargroups[t]], name = setdomain[m], mode='markers', legendgroup = str(t+1)), row=t+1, col=1)
        if lineargroups[t] == 'disorder_score':
            setss=list(set(plotdf["Secondary Structure Detailed"]))
            for m in range(len(setss)):
                ssubset=plotdf[plotdf["Secondary Structure Detailed"].isin([setss[m]])]
                fig.add_trace(go.Scatter(x=ssubset["position"], y=ssubset[lineargroups[t]], name = setss[m], mode='markers', legendgroup = str(t+1)), row=t+1, col=1)
        fig.update_yaxes(title_text=lineargroups[t], row=t+1, col=1)
    fig.update_layout(height=275*len(lineargroups), width=1100, legend_tracegroupgap = 180, title_text=transcriptset[k], colorway=px.colors.qualitative.G10)
    fig.update_xaxes(range=[0,max(list(position))+1],showgrid=False,showticklabels=False)
    fig.update_xaxes(title_text='Position', showticklabels=True, row=len(lineargroups), col=1)
    eold=exon[0]
    for e in range(len(exon)):
        if exon[e]!=eold:
            fig.add_vline(x=(position[e]+position[e-1])/2, line_dash="dot", row='all', col=1, line_color="#000000", line_width=2)
            eold=exon[e]
    fig.write_html('gdrive/MyDrive/CRISPRtile/output/'+transcriptset[k]+' linear tracks interactive.html')

# **Drug search on output of CRISPRtile**

The first step is to change runtime type in the upper right to GPU to speed up the analysis. The cell below sets up the folder for analysis with example input files.

In [ ]:
!git clone https://github.com/jasoncngo/BBBShinobi.git
%cd BBBShinobi
!pip install colabvenv
from colabvenv import install_python, create_env, run_in_env, run_python_in_env
install_python()
create_env('tcpi')
run_in_env('pip install pandas==1.3.4 torch==1.9 tape-proteins==0.5 rdkit==2022.3.3 numpy==1.19.5 scikit-learn==0.24.1 plotly','tcpi')
!gdown 1WL6S27O3BfObZgxnU1cOwf1bPG-Zg1Dn
import subprocess
venv_pip = "/content/BBBShinobi/tcpi/bin/pip"
subprocess.run([venv_pip, "install", "cffi"], check=True)
subprocess.run(["apt-get", "install", "-y", "liblmdb-dev"], check=True)
subprocess.run(["apt-get", "install", "-y", "python3.8-dev"], check=True)
import shutil, glob
for f in glob.glob("/content/BBBShinobi/tcpi/lib/python3.8/site-packages/lmdb/__pycache__/*"):
    print("Removing:", f)
    shutil.rmtree(f, ignore_errors=True)
subprocess.run([
    venv_pip, "install", "--force-reinstall", "--no-binary", ":all:", "lmdb==0.98"
], check=True)

Cloning into 'BBBShinobi'...
remote: Enumerating objects: 1202, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 1202 (delta 45), reused 1 (delta 1), pack-reused 1119 (from 1)
Receiving objects: 100% (1202/1202), 739.91 MiB | 25.56 MiB/s, done.
Resolving deltas: 100% (383/383), done.
Updating files: 100% (317/317), done.
/content/BBBShinobi
Command succeeded: python3 --version
Output:
Python 3.12.13

Command succeeded: sudo dpkg --configure -a
Output:

Command succeeded: sudo apt install software-properties-common -y
Output:
Reading package lists...
Building dependency tree...
Reading state information...
software-properties-common is already the newest version (0.99.22.9).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.

Command succeeded: yes '' | sudo add-apt-repository ppa:deadsnakes/ppa
Output:
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jamm

KeyboardInterrupt: 

An up to date list of FDA approved drugs can be obtained from Chembl using the code below. Alternatively, you can use your own custom list.

In [ ]:
import requests
import pandas as pd
import time
base_url = "https://www.ebi.ac.uk/chembl/api/data/molecule"
limit = 1000
offset = 0
all_records = []
while True:
    url = f"{base_url}?max_phase=4&limit={limit}&offset={offset}&format=json"
    response = requests.get(url)
    data = response.json()
    molecules = data['molecules']
    if not molecules:
        break

    for mol in molecules:
        st = mol.get('molecule_structures')
        if st and st.get('canonical_smiles'):
            all_records.append({
                'name': mol.get('pref_name'),
                'chembl_id': mol.get('molecule_chembl_id'),
                'smiles': st['canonical_smiles']
            })

    print(f"Fetched {len(molecules)} molecules (offset={offset})")
    offset += limit
    time.sleep(0.2)
df = pd.DataFrame(all_records)
df.to_csv("chembl_fda_drugs_smiles_all.csv", index=False)
print(f"Total FDA-approved molecules fetched: {len(df)}")

Example input files are smiles.csv and sequence.csv, you can download them to see the format and reupload them in the BBBShinobi folder with the sequence and compounds you want to test. The smiles.csv is already preloaded with FDA approved compounds for drug repurposing. The cell below calculates the score from the list of compounds and sequences. The output should be a screen csv file.

In [ ]:
run_in_env('python screen.py','tcpi')

The cell below does mutagenesis for each amino acid and calculates the score. The output should be a mutagenesis csv and html file.

In [ ]:
run_in_env('python computationalmutagenesis.py','tcpi')

# **Brain penetration prediction**

The cell below sets up the brain penetration prediction model

In [ ]:
!git clone https://github.com/jasoncngo/BBBShinobi.git
!pip install mordredcommunity
!pip install rdkit
!pip install autogluon.tabular[all]==1.3.1
!pip install fasttransform

An example smiles.csv file will be in the BBBShinobi folder. Download the smiles.csv file and put in your compounds following the same format as the sample file and run the cell below to predict the logBB. The output will be found in the BBBShinobi folder once it is finished.

In [ ]:
!python BBBShinobi/model/predict.py